Nikolaj Skou-Larsen - gkm406

In [29]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from maketables import dtable
from lets_plot import *
LetsPlot.setup_html()
df = pd.read_stata("A1_kommune.dta")

df.columns


Index(['nr', 'kommune', 'taxrev', 'taxrate', 'pop'], dtype='str')

Describtive analysis taxrev total

In [30]:
#Describtive analysis
vars =['taxrev','taxrate','pop']
print(df.groupby('kommune')[vars].mean().round(3))

# Max/min taxrev and taxrate
max_tax_rev = df.loc[df['taxrev'].idxmax()]

min_tax_rev = df.loc[df['taxrev'].idxmin()]

max_tax_rate = df.loc[df['taxrate'].idxmax()]

min_tax_rate = df.loc[df['taxrate'].idxmin()]

selected = pd.DataFrame([
    max_tax_rev,
    min_tax_rev,
    max_tax_rate,
    min_tax_rate
])
selected = selected.rename(columns={
    'taxrev': 'taxrev (mio. DKK)',
    'taxrate': 'taxrate (%)',
    'pop': 'population'
})
selected['kommune'] = [
    'Max tax revenue: ' + str(max_tax_rev['kommune']),
    'Min tax revenue: ' + str(min_tax_rev['kommune']),
    'Max tax rate: ' + str(max_tax_rate['kommune']),
    'Min tax rate: ' + str(min_tax_rate['kommune'])
]
selected['kommune'] = selected['kommune'].str.replace(' Kommune', '', regex=False)



dtable.DTable(
    selected,
    ['taxrev (mio. DKK)', 'taxrate (%)', 'population'],
    counts_row_below=False,
    bycol=['kommune'],
    stats=['mean'],
    caption="Table 1 - Descriptive Statistics"
)

                               taxrev    taxrate       pop
kommune                                                   
Aabenraa Kommune          4547.832031  25.400000   59978.0
Aalborg Kommune          16330.091797  25.400000  197426.0
Aarhus Kommune           26152.337891  24.400000  306650.0
Albertslund Kommune       2964.559082  24.600000   27730.0
Allerød Kommune           1613.119995  25.299999   24089.0
...                               ...        ...       ...
Vejle Kommune             8517.554688  23.400000  106383.0
Vesthimmerlands Kommune   3324.785889  27.200001   38106.0
Viborg Kommune            6772.346191  25.799999   93310.0
Vordingborg Kommune       3733.774902  24.900000   46319.0
Ærø Kommune                566.492981  26.100000    6679.0

[98 rows x 3 columns]


<maketables.mtable.MTable.__repr__.<locals>.DualOutput at 0x199a2e07380>

Describtive analysis taxrev per person

In [31]:

#Describtive analysis
vars =['taxrev','taxrate','pop']
print(df.groupby('kommune')[vars].mean().round(3))
df['taxrev_per_person'] = (df['taxrev'] * 1_000_000) / df['pop']


# Max/min taxrev and taxrate
max_tax_rev = df.loc[df['taxrev_per_person'].idxmax()]
min_tax_rev = df.loc[df['taxrev_per_person'].idxmin()]
max_tax_rate = df.loc[df['taxrate'].idxmax()]
min_tax_rate = df.loc[df['taxrate'].idxmin()]

# Creats dataframe for only values in Table
selected = pd.DataFrame([
    max_tax_rev,
    min_tax_rev,
    max_tax_rate,
    min_tax_rate
])
# Renames coloumns for readablitty 
selected = selected.rename(columns={
    'taxrev': 'tax revenue per person',
    'taxrate': 'taxrate (%)',
    'pop': 'population'
})
# Adds which category qualifies it for the tabel
selected['kommune'] = [
    'Max revenue: ' + str(max_tax_rev['kommune']),
    'Min revenue: ' + str(min_tax_rev['kommune']),
    'Max rate: ' + str(max_tax_rate['kommune']),
    'Min rate: ' + str(min_tax_rate['kommune'])
]
# Removes Kommune from the table
selected['kommune'] = selected['kommune'].str.replace(' Kommune', '', regex=False)

#Creates the Table
dtable.DTable(
    selected,
    ['tax revenue per person', 'taxrate (%)', 'population'],
    counts_row_below=True,
    bycol=['kommune'],
    stats=['mean'],
    caption="Table 2 - Maximum & minimum values"
)



                               taxrev    taxrate       pop
kommune                                                   
Aabenraa Kommune          4547.832031  25.400000   59978.0
Aalborg Kommune          16330.091797  25.400000  197426.0
Aarhus Kommune           26152.337891  24.400000  306650.0
Albertslund Kommune       2964.559082  24.600000   27730.0
Allerød Kommune           1613.119995  25.299999   24089.0
...                               ...        ...       ...
Vejle Kommune             8517.554688  23.400000  106383.0
Vesthimmerlands Kommune   3324.785889  27.200001   38106.0
Viborg Kommune            6772.346191  25.799999   93310.0
Vordingborg Kommune       3733.774902  24.900000   46319.0
Ærø Kommune                566.492981  26.100000    6679.0

[98 rows x 3 columns]


<maketables.mtable.MTable.__repr__.<locals>.DualOutput at 0x199a2e07380>

# Problem 2

##### First OLS

In [32]:
#OLS for log-level
df = pd.read_stata("A1_kommune.dta")
df['log_taxrev'] = np.log(df.taxrev)


df['const'] = 1


result = sm.OLS(df['log_taxrev'], df[['const','taxrate']]).fit()

print(result.summary())  
ols_table = pd.DataFrame({
    'Coefficient': result.params,
    'Std. Error': result.bse
})

ols_table.index = ['δ₀', 'δ₁']

print(ols_table.round(3))

                            OLS Regression Results                            
Dep. Variable:             log_taxrev   R-squared:                       0.029
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     2.818
Date:                Sun, 13 Sep 2026   Prob (F-statistic):             0.0965
Time:                        15:03:07   Log-Likelihood:                -111.12
No. Observations:                  98   AIC:                             226.2
Df Residuals:                      96   BIC:                             231.4
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         11.6982      2.143      5.459      0.0

Negativ beta1 kan type på at højere skat får folk til at arbejde mindere og dermed bliver skatteindtægterne lavere. Laffer kurven? 

#### Second OLS

In [43]:
def OLS(X,y):
    betahat = (np.linalg.inv(X.T @ X)) @ X.T @ y
    for i in range(len(betahat)):
        print(f'betahat{i} = {betahat[i]}')
    yhat =  X @ betahat
    y_bar = y.mean()
    uhat = y - yhat

    SST = np.sum((y - y_bar)**2)
    SSE = np.sum((yhat-y_bar)**2)
    SSR = np.sum(uhat**2)
    R2 = SSE / SST

    n, k_1 = X.shape
    k = k_1 - 1

    sigma2 = SSR / (n - k - 1)

    covarmatrix = sigma2 * np.linalg.inv(X.T @ X)
    se_betahat = np.sqrt(np.diag(covarmatrix))

    return{
        'SST' : SST.round(3),
        'SSE' : SSE.round(3),
        'SSR' : SSR.round(3),
        'R2': R2.round(3),
        'sigma2': sigma2.round(3),
        'se(betahat)': se_betahat.round(3)
           }

df['log_pop'] = np.log(df['pop'])

X = np.column_stack([
    np.ones(len(df)),
    df['taxrate'].values,
    df['log_pop'].values
])
y = df['log_taxrev'].values

result = OLS(X,y)

print(result)

betahat0 = -2.8021657237787423
betahat1 = 0.022622273090221634
betahat2 = 0.9711010677941873
{'SST': np.float32(57.042), 'SSE': np.float64(55.909), 'SSR': np.float64(1.133), 'R2': np.float64(0.98), 'sigma2': np.float64(0.012), 'se(betahat)': array([0.376, 0.012, 0.014])}


In [46]:
y = df['log_taxrev']

X = df[['taxrate', 'log_pop']]
X = sm.add_constant(X)

result = sm.OLS(y, X).fit()

print(result.summary())



                            OLS Regression Results                            
Dep. Variable:             log_taxrev   R-squared:                       0.980
Model:                            OLS   Adj. R-squared:                  0.980
Method:                 Least Squares   F-statistic:                     2344.
Date:                Sun, 13 Sep 2026   Prob (F-statistic):           1.42e-81
Time:                        15:09:31   Log-Likelihood:                 79.497
No. Observations:                  98   AIC:                            -153.0
Df Residuals:                      95   BIC:                            -145.2
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -2.8022      0.376     -7.461      0.0